#Installing libraries

In [1]:
!pip install -q chromadb==0.5.5
!pip install -q gradio==4.44.0
!pip install -q langchain-community==0.3.0
!pip install -qU langchain-groq
!pip install -q unstructured==0.15.0 unstructured[pdf]==0.15.0
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━

In [2]:
from langchain.document_loaders import UnstructuredFileLoader

loader = UnstructuredFileLoader("/content/sample_data/bula_paracetamol_10162_1318.pdf")
documents = loader.load()

<ipython-input-2-966c38fcfacf>:3: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-unstructured package and should be used instead. To use it run `pip install -U :class:`~langchain-unstructured` and import as `from :class:`~langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader("/content/sample_data/bula_paracetamol_10162_1318.pdf")


In [3]:
from langchain.text_splitter import CharacterTextSplitter

In [4]:
text_splitter = CharacterTextSplitter(chunk_size=1250,
                                      separator="\n",
                                      chunk_overlap=100)
text_chunks = text_splitter.split_documents(documents)


In [5]:
first_doc = text_chunks[1]
print(first_doc.page_content)

Indicações do medicamento: é indicado em adultos para a redução da febre e para o alívio temporário de dores leves a moderadas, tais como: dores associadas a gripes e resfriados comuns, dor de cabeça, dor de dente, dor nas costas, dores musculares, dores associadas a artrites e cólicas menstruais.
Riscos do medicamento: Contraindicações: você não deve tomar paracetamol se tiver hipersensibilidade (alergia) ao paracetamol ou aos outros componentes da fórmula.


# Initialize the Embedding Model and Vector DB

In [6]:
import os
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "sua chave"

In [7]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma.from_documents(text_chunks, embeddings, persist_directory="Vector_DB_directory")
print("Documents vectorized.")

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in HuggingFaceInferenceAPIEmbeddings has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
<ipython-input-7-4359b200192b>:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Documents vectorized.


In [8]:
persist_directory = "Vector_DB_directory"

In [9]:
# load the chroma db
crhomadb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)

<ipython-input-9-b75f0cff96ea>:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  crhomadb = Chroma(


In [10]:
from langchain_groq import ChatGroq  # Supondo que haja um wrapper para Groq
from langchain.chains.conversation.memory import ConversationBufferWindowMemory
from langchain.chains import RetrievalQA

# llm from groq
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    temperature=0
)

# Memória da conversa com buffer de 4 mensagens
conversational_memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    k=4,  # Número de mensagens armazenadas na memória
    return_messages=True  # Retorna as mensagens na resposta
)

# Configurando a cadeia de perguntas e respostas com recuperação (RetrievalQA)
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=crhomadb.as_retriever(),
    return_source_documents=False
)

We can try the isolated Retrieval to see if the information it returns is relevant.




In [11]:
qa.invoke("Dosagem")

{'query': 'Dosagem',
 'result': 'A dosagem recomendada para adultos e crianças de 12 anos ou mais é de 500 a 1000 mg/dose, com intervalos de 4 a 6 horas entre cada administração. Não deve exceder o total de 4 g em 24 horas.\n\nPara o paracetamol 750 mg, a recomendação é:\n\n* 1 comprimido, 3 a 5 vezes ao dia.\n* Não exceder 5 comprimidos, em doses fracionadas, em um período de 24 horas.'}

Perfect! The information returned is exactly what we desired.

# Creating the Agent.

In [12]:
from langchain.agents import Tool

#Defining the list of tool objects to be used by LangChain.
tools = [
    Tool(
        name='Pharmaceutical',
        func=qa.invoke,
        description=(
            """use this tool when answering pharmaceutical knowledge queries to get
            more information about the topic"""
        )
    )
]

In [13]:
from langchain.agents import create_react_agent
from langchain import hub

prompt = hub.pull("hwchase17/react-chat")

agent = create_react_agent(
    tools=tools,
    llm=llm,
    prompt=prompt,
)

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:323: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [14]:
# Create an agent executor by passing in the agent and tools
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               verbose=True,
                               memory=conversational_memory,
                               max_iterations=30,
                               max_execution_time=600,
                               handle_parsing_errors=True
                               )

## Using the Conversational Agent

In [15]:
agent_executor.invoke({"input": """Um paciente adulto tem dor de cabeça, qual é a dosagem de paracetamol?"""})

Thought: Do I need to use a tool? Yes
Action: Pharmaceutical
Action Input: {"question": "paracetamol dosage for adult patient with headache"}{'query': '{"question": "paracetamol dosage for adult patient with headache"}', 'result': 'De acordo com as informações fornecidas, a dosagem recomendada de paracetamol para adultos e crianças de 12 anos ou mais é de 500 a 1000 mg/dose, com intervalos de 4 a 6 horas entre cada administração. Não se deve exceder o total de 4 g em 24 horas.\n\nPara o comprimido de paracetamol 750 mg, a recomendação é de 1 comprimido, 3 a 5 vezes ao dia. Não se deve exceder 5 comprimidos em doses fracionadas em um período de 24 horas.\n\nPortanto, para um paciente adulto com dor de cabeça, a dosagem recomendada seria de 1 comprimido de paracetamol 750 mg, que pode ser repetida a cada 4 a 6 horas, não excedendo 5 comprimidos em 24 horas.'}Thought: Do I need to use a tool? No
Final Answer: A dosagem recomendada de paracetamol para um paciente adulto com dor de cabeça é

{'input': 'Um paciente adulto tem dor de cabeça, qual é a dosagem de paracetamol?',
 'chat_history': [],
 'output': 'A dosagem recomendada de paracetamol para um paciente adulto com dor de cabeça é de 1 comprimido de 750 mg, que pode ser repetida a cada 4 a 6 horas, não excedendo 5 comprimidos em 24 horas.'}

Perfect, the most important thing for us is that it has been able to identify that it should go to the medical database to search for information about the symptoms.

In [16]:
response = agent_executor.invoke({"input": "Um paciente criança tem dor de cabeça, qual é a dosagem de paracetamol?"})

Thought: Do I need to use a tool? Yes
Action: Pharmaceutical
Action Input: {"question": "paracetamol dosage for child with headache", "context": "child"}{'query': '{"question": "paracetamol dosage for child with headache", "context": "child"}', 'result': 'De acordo com as informações fornecidas, o paracetamol não deve ser utilizado em crianças abaixo de 12 anos de idade.'}Thought: Do I need to use a tool? No
Final Answer: De acordo com as informações fornecidas, o paracetamol não deve ser utilizado em crianças abaixo de 12 anos de idade. É importante consultar um médico antes de administrar qualquer medicamento a uma criança.

> Finished chain.


In [ ]:
response['output']

'De acordo com as informações fornecidas, o paracetamol não deve ser utilizado em crianças abaixo de 12 anos de idade. Portanto, não há uma dosagem recomendada para crianças com menos de 12 anos. É importante consultar um médico antes de administrar qualquer medicamento a uma criança.'

#Adding Gradio Interface
We are going to create an Interface with Gradio that can work with a LangChain Agent.





In [17]:
#import gradio library.
import gradio as gr
#history = ""

In [18]:
# Define the function for the conversation
def continue_conversation(input, history):
    # Invoke the agent and get the response
    response = agent_executor.invoke({"input": input})
    output = response['output']

    # Append the new input and output to the history
    history.append(f"Doctor: {input}")
    history.append(f"AI Assistant: {output}")

    # Join the history into a single string
    history_text = "\n".join(history)

    # Return the current response and the full history (hidden state)
    return output, history_text, history

In [19]:
#Function call by the clear button to clear the Input textBox.
def clear_input():
    return ""

In [20]:
# Create the Gradio interface
with gr.Blocks() as demo:
  with gr.Row():
    #We use two columns to organize the Gradio Elements.
    with gr.Column():
      # Input textbox
      input_textbox = gr.Textbox(lines=5, placeholder="Type your prompt here...")
      # Conversation history state
      history_state = gr.State([])

      # Outputs
      current_response = gr.Textbox(label="Current Response")
      conversation_history = gr.Textbox(label="Conversation History", lines=10)
    with gr.Column():
    # Buttons
      send_button = gr.Button("Send")
      clear_button = gr.Button("Clear Input")

      # Bind the send button to submit the input
      send_button.click(
          fn=continue_conversation,
          inputs=[input_textbox, history_state],
          outputs=[current_response, conversation_history, history_state]
      )

      # Bind the clear button to clear the input
      clear_button.click(fn=clear_input, inputs=[], outputs=[input_textbox])

      # Alternatively, pressing "Enter" in the input box will also submit
      input_textbox.submit(
          fn=continue_conversation,
          inputs=[input_textbox, history_state],
          outputs=[current_response, conversation_history, history_state]
      )

In [ ]:
demo.launch(debug=True)

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://4f8d0f42879943d4ea.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Thought: Do I need to use a tool? No
Final Answer: A dosagem recomendada de paracetamol para um paciente adulto com dor de cabeça é de 1 comprimido de 750 mg, que pode ser repetida a cada 4 a 6 horas, não excedendo 5 comprimidos em 24 horas.

> Finished chain.


Thought: Do I need to use a tool? No
Final Answer: De acordo com as informações fornecidas, o paracetamol não deve ser utilizado em crianças abaixo de 12 anos de idade. É importante consultar um médico antes de administrar qualquer medicamento a uma criança.

> Finished chain.


Thought: Do I need to use a tool? Yes
Action: Pharmaceutical
Action Input: {"medicamento": "paracetamol", "efeitos_colaterais": True}{'query': '{"medicamento": "paracetamol", "efeitos_colaterais": True}', 'result': 'Sim, o paracetamol pode causar efeitos colaterais. De acordo com as informações fornecidas, podem ocorrer reações adversas inesperadas, incluindo reações de sensibilidade. Além disso, o uso crônico de bebidas alcoólicas pode aumentar o risco de doenças do fígado se for ingerida uma dose maior que a dose recomendada de paracetamol.'}Thought: Do I need to use a tool? No
Final Answer: O paracetamol pode causar efeitos colaterais, incluindo reações adversas inesperadas, como reações de sensibilidade. Além disso, o uso crônico de bebidas alcoólicas pode aumentar o risco de doenças do fígado se for ingerida uma dose maior que a dose recomendada de paracetamol.

> Finished chain.


Thought: Do I need to use a tool? Yes
Action: Pharmaceutical
Action Input: {"query": "superdose de paracetamol"}{'query': '{"query": "superdose de paracetamol"}', 'result': 'Uma superdose de paracetamol pode causar hepatotoxicidade em alguns pacientes. Em adultos e adolescentes, pode ocorrer hepatotoxicidade após a ingestão de mais que 7,5 a 10 g em um período de 8 horas ou menos. Em caso de suspeita de ingestão de altas doses de paracetamol, é importante procurar imediatamente um centro médico de urgência.\n\nOs sinais e sintomas iniciais que se seguem a uma dose potencialmente hepatotóxica de paracetamol incluem:\n\n* Náusea\n* Vômito\n* Sudorese intensa\n* Mal-estar geral\n\nOs sinais clínicos e laboratoriais de toxicidade hepática podem não estar presentes até 48 a 72 horas após a ingestão da dose maciça.\n\nO tratamento para superdose de paracetamol inclui:\n\n* Esvaziamento do estômago por lavagem gástrica ou indução ao vômito com xarope de ipeca\n* Administração de N-acetilciste

---